# Transformación de Datos

Limpia, normaliza, reorganiza y enriquece datos para análisis

## Introducción

La transformación es el corazón de cualquier pipeline ETL: convertir datos crudos, sucios e inconsistentes en datos limpios, estructurados y listos para análisis. Incluye limpieza (nulos, duplicados, tipos), normalización (escalado, estandarización), transformaciones de texto, ingeniería de características (columnas calculadas, binning, encoding), y reshaping (pivotar, fundir, agregar). Dominar estas técnicas con pandas es la diferencia entre un dataset utilizable y uno que distorsiona el análisis.

### Objetivos de Aprendizaje

- Construir un pipeline de limpieza: nulos, duplicados y conversión de tipos
- Normalizar y estandarizar columnas numéricas con pandas
- Transformar y estandarizar columnas de texto con el accessor .str y regex
- Crear columnas calculadas y aplicar ingeniería de características básica
- Reshapear datos con pivot_table, melt y groupby para cambiar la estructura

## Pipeline de Limpieza: Nulos, Duplicados y Tipos

> El orden importa en un pipeline de limpieza. El orden correcto es: 1) Eliminar duplicados (antes de imputar para no multiplicar errores), 2) Convertir tipos (necesario para que las operaciones posteriores funcionen), 3) Imputar nulos (con valores calculados correctamente sobre el tipo adecuado), 4) Limpiar strings. Saltar pasos o hacerlos en orden incorrecto produce resultados incorrectos o errores difíciles de diagnosticar.

In [ ]:
import pandas as pd
import numpy as np

datos_empleados = pd.DataFrame({
    'id':          [1, 2, 2, 3, 4, 5, 5],
    'nombre':      ['Ana García', 'Carlos', 'Carlos', 'María', None, 'Luis', 'Luis'],
    'salario':     ['48000', '52000', '52000', '61500', 'N/A', '43000', '43000'],
    'antiguedad':  [3.0, 7.0, 7.0, 12.0, None, 5.0, 5.0],
    'activo':      ['True', 'True', 'True', 'False', 'True', 'True', 'True']
})

print(f"Filas originales: {len(datos_empleados)}")
print(datos_empleados, "\n")

df = datos_empleados.drop_duplicates()
print(f"Tras drop_duplicates(): {len(df)} filas")

df = df.drop_duplicates(subset=['id'], keep='first')
print(f"Tras deduplicar por ID: {len(df)} filas")

df['salario'] = pd.to_numeric(df['salario'], errors='coerce')
df['activo'] = df['activo'].map({'True': True, 'False': False})

df['salario'] = df['salario'].fillna(df['salario'].median())
df['antiguedad'] = df['antiguedad'].fillna(df['antiguedad'].mean().round(1))
df['nombre'] = df['nombre'].fillna('Desconocido')

print("\nDataset limpio:")
print(df)
print("\nTipos de datos:")
print(df.dtypes)

## Normalización y Estandarización con pandas

> La normalización escala los valores a un rango [0,1] — útil para algoritmos que usan distancias (k-NN, redes neuronales). La estandarización (z-score) centra los datos en media=0 y desvío=1 — mejor para algoritmos que asumen distribución normal (regresión, SVM). pandas permite implementar ambas sin dependencias externas; sklearn las ofrece como transformadores reutilizables para producción.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'empleado':  ['Ana', 'Carlos', 'María', 'Luis', 'Elena'],
    'salario':   [48000, 52000, 61500, 43000, 55000],
    'antiguedad': [3, 7, 12, 5, 9],
    'ventas':    [120000, 85000, 210000, 65000, 175000]
})

print("=== DATOS ORIGINALES ===")
print(df[['salario', 'antiguedad', 'ventas']].describe().round(0))

def normalizar_minmax(serie: pd.Series) -> pd.Series:
    return (serie - serie.min()) / (serie.max() - serie.min())

df['salario_norm']    = normalizar_minmax(df['salario'])
df['antiguedad_norm'] = normalizar_minmax(df['antiguedad'])
df['ventas_norm']     = normalizar_minmax(df['ventas'])

print("\n=== NORMALIZACIÓN MIN-MAX (rango 0-1) ===")
print(df[['empleado','salario_norm','antiguedad_norm','ventas_norm']].round(3))

def estandarizar_zscore(serie: pd.Series) -> pd.Series:
    return (serie - serie.mean()) / serie.std()

df['salario_z']    = estandarizar_zscore(df['salario'])
df['antiguedad_z'] = estandarizar_zscore(df['antiguedad'])
df['ventas_z']     = estandarizar_zscore(df['ventas'])

print("\n=== ESTANDARIZACIÓN Z-SCORE (media≈0, std≈1) ===")
print(df[['empleado','salario_z','antiguedad_z','ventas_z']].round(3))

print(f"\nSalario z — media: {df['salario_z'].mean():.6f}, std: {df['salario_z'].std():.3f}")

## Transformaciones de Texto con .str y Regex

> El accessor .str de pandas vectoriza operaciones de string sobre toda una columna sin loops. Operaciones fundamentales: strip() elimina espacios, lower()/upper() cambia capitalización, replace() sustituye con soporte de regex, split() divide, extract() extrae grupos con regex, contains() filtra. Las expresiones regulares (regex) son esenciales para patrones complejos: extraer códigos, limpiar formatos, validar datos.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'nombre_completo': ['  ANA GARCÍA LÓPEZ  ', 'carlos martín', 'MARÍA DE LA FUENTE', '  Ángel'],
    'email':           ['ANA@empresa.COM', 'CARLOS@empresa.com', 'maria.fuente@EMPRESA.ES', 'angel@otra.org'],
    'telefono':        ['(+34) 612-345-678', '+34 612.345.679', '0034-612 345 680', '612345681'],
    'codigo_empleado': ['EMP-2024-0001', 'EMP-2023-0145', 'EMP-2022-0892', 'EMP-2024-0234'],
    'descripcion':     ['  Analista de datos senior  ', 'JEFE DE VENTAS', 'Ingeniería de software', None]
})

df['nombre'] = df['nombre_completo'].str.strip().str.title()
df['email']  = df['email'].str.lower().str.strip()

df['tel_limpio'] = df['telefono'].str.replace(r'[()+-.]', '', regex=True)
df['tel_limpio'] = df['tel_limpio'].str.replace(r'^(0034|34)', '', regex=True)

partes = df['codigo_empleado'].str.extract(r'EMP-(\d{4})-(\d{4})')
df['anio_contrato'] = partes[0].astype(int)
df['num_empleado']  = partes[1].astype(int)

senior_mask = df['descripcion'].str.contains(r'senior|jefe', case=False, na=False)
df_senior = df[senior_mask]

df['descripcion_limpia'] = df['descripcion'].str.strip().str.capitalize()
df['descripcion_limpia'] = df['descripcion_limpia'].fillna('Sin descripción')

print(df[['nombre', 'email', 'tel_limpio', 'anio_contrato', 'num_empleado',
          'descripcion_limpia']].to_string(index=False))

## Columnas Calculadas e Ingeniería de Características

> La ingeniería de características (feature engineering) crea nuevas columnas derivadas de las existentes para enriquecer el análisis o mejorar modelos de ML. Incluye: columnas calculadas simples (precio * cantidad), discretización/binning (edad → grupo_etario), encoding de categóricas (get_dummies para one-hot), extracción de partes de fecha (año, mes, día semana), y features de contexto (diferencia respecto a la media del grupo).

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'producto':   ['Laptop', 'Mouse', 'Monitor', 'Teclado', 'Laptop', 'Mouse', 'Monitor'],
    'categoria':  ['Electrónica', 'Periféricos', 'Electrónica', 'Periféricos',
                   'Electrónica', 'Periféricos', 'Electrónica'],
    'precio':     [1200.0, 25.5, 350.0, 45.0, 1150.0, 22.0, 380.0],
    'costo':      [800.0, 12.0, 200.0, 20.0, 750.0, 10.0, 210.0],
    'unidades':   [5, 50, 8, 30, 7, 60, 6],
    'fecha':      pd.to_datetime(['2024-01-15','2024-01-15','2024-01-16',
                                 '2024-01-16','2024-02-10','2024-02-10','2024-02-11'])
})

df['margen_unit']  = (df['precio'] - df['costo']).round(2)
df['margen_pct']   = ((df['margen_unit'] / df['precio']) * 100).round(1)
df['ingreso_total']= (df['precio'] * df['unidades']).round(2)
df['beneficio']    = (df['margen_unit'] * df['unidades']).round(2)

df['anio']       = df['fecha'].dt.year
df['mes']        = df['fecha'].dt.month
df['semana']     = df['fecha'].dt.isocalendar().week.astype(int)
df['dia_semana'] = df['fecha'].dt.day_name()

bins   = [0, 50, 200, 500, float('inf')]
labels = ['Económico', 'Medio', 'Premium', 'Ultra-Premium']
df['rango_precio'] = pd.cut(df['precio'], bins=bins, labels=labels)

media_cat = df.groupby('categoria')['precio'].transform('mean')
df['precio_vs_media_cat'] = (df['precio'] - media_cat).round(2)

df_encoded = pd.get_dummies(df, columns=['categoria'], prefix='cat', dtype=int)

print(df[['producto','margen_pct','ingreso_total','rango_precio',
          'dia_semana','precio_vs_media_cat']].to_string(index=False))

## Reshaping: pivot_table, melt y Agregaciones

> Reshaping cambia la estructura del DataFrame sin perder datos. pivot_table convierte filas en columnas (de formato largo a ancho), ideal para tablas de contingencia y reportes. melt hace lo inverso (de ancho a largo), necesario para visualizaciones y cuando las columnas representan categorías. groupby + agg genera resúmenes estadísticos. La elección entre ancho y largo depende del análisis que necesites hacer.

In [ ]:
import pandas as pd

ventas = pd.DataFrame({
    'fecha':      ['2024-01','2024-01','2024-01','2024-02','2024-02','2024-02'],
    'region':     ['Norte', 'Sur', 'Este', 'Norte', 'Sur', 'Este'],
    'producto':   ['Laptop','Laptop','Mouse','Laptop','Mouse','Laptop'],
    'ventas':     [12000, 8500, 2500, 15000, 3200, 9000],
    'unidades':   [10, 7, 100, 12, 130, 8]
})

tabla_pivot = pd.pivot_table(
    ventas,
    values='ventas',
    index='region',
    columns='fecha',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='TOTAL'
)
print("=== PIVOT: Ventas por Región y Mes ===")
print(tabla_pivot)

tabla_multi = pd.pivot_table(
    ventas,
    values=['ventas', 'unidades'],
    index='region',
    columns='fecha',
    aggfunc={'ventas': 'sum', 'unidades': 'sum'},
    fill_value=0
)
print("\n=== PIVOT MULTI-MÉTRICA ===")
print(tabla_multi)

df_ancho = pd.DataFrame({
    'producto': ['Laptop', 'Mouse', 'Monitor'],
    'Q1_ventas': [45000, 8500, 21000],
    'Q2_ventas': [52000, 9200, 19000],
    'Q3_ventas': [48000, 7800, 23000],
    'Q4_ventas': [61000, 11000, 28000]
})

df_largo = df_ancho.melt(
    id_vars=['producto'],
    value_vars=['Q1_ventas','Q2_ventas','Q3_ventas','Q4_ventas'],
    var_name='trimestre',
    value_name='ventas'
)
df_largo['trimestre'] = df_largo['trimestre'].str.replace('_ventas', '')
print("\n=== MELT: Ancho → Largo ===")
print(df_largo.to_string(index=False))

resumen = (
    ventas
    .groupby('region')
    .agg(
        total_ventas  =('ventas', 'sum'),
        venta_promedio=('ventas', 'mean'),
        total_unidades=('unidades', 'sum'),
        num_transacc  =('ventas', 'count')
    )
    .round(0)
    .sort_values('total_ventas', ascending=False)
)
print("\n=== GROUPBY + AGG ===")
print(resumen)

## Pipeline Completo de Transformación de Datos de Ventas

Aplica todas las transformaciones necesarias a un dataset de ventas: limpieza, tipos, features calculadas y reshaping para análisis.

In [ ]:
import pandas as pd
import numpy as np
import io

CSV_VENTAS = """id,fecha,vendedor,producto,categoria,precio,costo,unidades,region,notas
1,15/01/2024,Ana García,Laptop Pro,ELECTRÓNICA,1200,800,3,norte,Venta normal
2,15/01/2024,Carlos López,Mouse WL,periféricos,25.5,12,10,SUR,Pedido urgente
3,16/01/2024,Ana García,Monitor 27,Electrónica,350,200,2,Norte,
3,16/01/2024,Ana García,Monitor 27,Electrónica,350,200,2,Norte,
4,16/01/2024,,Teclado Mec,PERIFÉRICOS,45,20,5,ESTE,Sin vendedor
5,17/01/2024,María Ruiz,Laptop Pro,Electrónica,N/A,800,1,sur,Precio pendiente
6,17/01/2024,Carlos López,Monitor 27,Electrónica,380,210,3,Sur,
7,2024-01-18,Ana García,Mouse WL,Periféricos,22,10,15,norte,"""

def transformar_ventas(csv_texto: str) -> pd.DataFrame:
    df = pd.read_csv(io.StringIO(csv_texto))
    print(f"[1] Cargado: {len(df)} filas brutas")

    df.drop_duplicates(inplace=True)
    print(f"[2] Tras drop_duplicates: {len(df)} filas")

    df['categoria'] = df['categoria'].str.strip().str.title()
    df['region']    = df['region'].str.strip().str.title()
    df['vendedor']  = df['vendedor'].str.strip().str.title()
    
    df.drop_duplicates(subset=['id'], keep='first', inplace=True)
    print(f"[3] Tras deduplicar por ID: {len(df)} filas")

    df['precio']   = pd.to_numeric(df['precio'], errors='coerce')
    df['costo']    = pd.to_numeric(df['costo'], errors='coerce')
    df['unidades'] = pd.to_numeric(df['unidades'], errors='coerce').astype('Int64')

    df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')

    df['precio']   = df.groupby('producto')['precio'].transform(
        lambda x: x.fillna(x.median())
    )
    df['vendedor'] = df['vendedor'].fillna('Sin Asignar')
    df['notas']    = df['notas'].fillna('')

    df['margen']       = (df['precio'] - df['costo']).round(2)
    df['margen_pct']   = ((df['margen'] / df['precio']) * 100).round(1)
    df['ingreso_total']= (df['precio'] * df['unidades']).round(2)
    df['beneficio']    = (df['margen'] * df['unidades']).round(2)
    df['mes']          = df['fecha'].dt.to_period('M').astype(str)
    df['dia_semana']   = df['fecha'].dt.day_name()

    bins   = [0, 50, 200, 600, float('inf')]
    etiquetas = ['Básico', 'Medio', 'Alto', 'Premium']
    df['rango_precio'] = pd.cut(df['precio'], bins=bins, labels=etiquetas)

    print(f"[4] Transformación completa: {len(df)} filas limpias")
    return df

df_limpio = transformar_ventas(CSV_VENTAS)
cols = ['fecha', 'vendedor', 'producto', 'categoria', 'region',
        'precio', 'margen_pct', 'ingreso_total', 'rango_precio']
print("\n=== DATOS TRANSFORMADOS ===")
print(df_limpio[cols].to_string(index=False))

tabla = pd.pivot_table(
    df_limpio, values='ingreso_total',
    index='region', columns='categoria',
    aggfunc='sum', fill_value=0, margins=True
)
print("\n=== PIVOT: Ingresos por Región y Categoría ===")
print(tabla)

## Tips y Mejores Prácticas

> Siempre elimina duplicados ANTES de imputar valores nulos. Si imputa primero y hay duplicados, los valores imputed se multiplican y afectan las estadísticas de la imputación (la media, mediana calculadas incluirán los duplicados).

> pd.get_dummies() crea tantas columnas nuevas como valores únicos tenga la categoría. Si una columna tiene 1000 categorías distintas, generará 1000 nuevas columnas. Para alta cardinalidad, considera target encoding o frequency encoding en su lugar.

> Usa df.groupby("grupo")["col"].transform("mean") para crear features contextuales (valor vs. media del grupo) sin necesidad de hacer un merge. transform() devuelve una Series del mismo tamaño que el DataFrame original.

> melt() es tu aliado cuando tienes columnas que representan categorías (Q1_ventas, Q2_ventas, Q3_ventas). Convierte ese formato "ancho" a "largo" que es el esperado por seaborn, plotly y la mayoría de herramientas de visualización y modelos.

## Errores Comunes

### Normalizar antes de separar los datos en train/test

¿Por qué ocurre?
- Si normalizas con min/max de todo el dataset (incluyendo el test set), estás "filtrando" información del futuro al pasado. El scaler debe ajustarse SOLO en el training set y aplicarse al test set.

Solución
- En producción, guarda los parámetros de normalización (min, max, media, std) calculados en train, y úsalos para transformar el test set y nuevos datos.

### Usar pivot_table cuando los datos ya están en el formato correcto

¿Por qué ocurre?
- Muchos principiantes intentan pivotar datos que ya están bien estructurados, o pivotan cuando lo que necesitan es un simple groupby. Esto añade complejidad innecesaria.

Solución
- Usa pivot_table cuando necesites una tabla de doble entrada (fila × columna). Para resúmenes simples de una dimensión, groupby().agg() es más claro y eficiente.

### Modificar una columna de string en su lugar sin asignar el resultado

¿Por qué ocurre?
- df["col"].str.strip() devuelve una nueva Series — no modifica el DataFrame. Muchos principiantes esperan que la modificación sea in-place.

Solución
- Siempre asigna el resultado: df["col"] = df["col"].str.strip().str.lower().

### Aplicar pd.cut() con intervalos que no cubren todos los valores

¿Por qué ocurre?
- Si los datos tienen valores fuera del rango de los bins definidos, esos valores quedan como NaN.

Solución
- Siempre usa 0 (o -inf) como límite inferior y float("inf") como límite superior en tus bins. Verifica con df["nueva_col"].isna().sum() que no haya NaN inesperados.